# Rebar YOLO26L Segmentation Visualization

This notebook loads reusable visualization utilities first, then reviews each training version with its training info, curves, and predicted mask overlays.


## Setup and Utilities


In [ ]:
from pathlib import Path
import csv
import os
import re

ROOT = Path.cwd()
if ROOT.name == "rebar-segementation-yolo26":
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

try:
    import yaml
except ImportError:
    yaml = None

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)
from PIL import Image


In [ ]:
def load_yaml(path):
    path = Path(path)
    if yaml is None:
        raise ImportError("PyYAML is required to read training and dataset YAML files.")
    with path.open("r") as f:
        return yaml.safe_load(f) or {}


def parse_roboflow_readme(path):
    path = Path(path)
    if not path.exists():
        return {}

    text = path.read_text()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    info = {"title": lines[0] if lines else None}

    exported = re.search(r"exported via roboflow.com on (.+)", text)
    if exported:
        info["exported"] = exported.group(1).strip()

    image_count = re.search(r"dataset includes (\d+) images", text, flags=re.IGNORECASE)
    if image_count:
        info["images"] = int(image_count.group(1))

    annotation = re.search(r"Intersection are annotated in ([^.]+)\.", text)
    if annotation:
        info["annotation_format"] = annotation.group(1).strip()

    return info


def read_results_metrics(path):
    path = Path(path)
    if not path.exists():
        return {}

    with path.open("r", newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}

    def number(row, key):
        try:
            return float(row[key])
        except (KeyError, TypeError, ValueError):
            return None

    final = rows[-1]
    mask_map_key = "metrics/mAP50-95(M)"
    best_row = max(rows, key=lambda row: number(row, mask_map_key) if number(row, mask_map_key) is not None else -1)

    return {
        "epochs_recorded": int(float(final.get("epoch", 0))),
        "final_mask_map50": number(final, "metrics/mAP50(M)"),
        "final_mask_map50_95": number(final, mask_map_key),
        "best_mask_map50_95": number(best_row, mask_map_key),
        "best_mask_map50_95_epoch": int(float(best_row.get("epoch", 0))),
        "final_box_map50_95": number(final, "metrics/mAP50-95(B)"),
    }


def format_value(value):
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(v) for v in value)
    return str(value)


def display_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except ImportError:
        display(rows)


In [ ]:
def overlay_masks(image, masks):
    image = image.convert("RGBA")
    if masks is None or len(masks) == 0:
        return image

    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [tuple(int(c * 255) for c in cmap(i)[:3]) for i in range(n_masks)]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        if mask.size != image.size:
            mask = mask.resize(image.size, Image.Resampling.NEAREST)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image


def validate_version_paths(version):
    paths = [
        version["run_dir"],
        version["model_path"],
        version["results_image"],
        version["results_csv"],
        version["args_yaml"],
        version["data_yaml"],
        version["valid_images"],
    ]
    missing = [path for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(str(path) for path in missing))


def validation_image_paths(version):
    valid_images = Path(version["valid_images"])
    return sorted(p for p in valid_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})


def display_training_info(version):
    validate_version_paths(version)

    args = load_yaml(version["args_yaml"])
    data = load_yaml(version["data_yaml"])
    roboflow = parse_roboflow_readme(version["roboflow_readme"])
    metrics = read_results_metrics(version["results_csv"])

    rows = [
        {"Field": "Version", "Value": version["version"]},
        {"Field": "Run", "Value": version["run_dir"].name},
        {"Field": "Dataset", "Value": version["dataset_dir"].relative_to(ROOT)},
        {"Field": "Dataset README", "Value": version["roboflow_readme"].relative_to(ROOT)},
        {"Field": "Roboflow title", "Value": roboflow.get("title")},
        {"Field": "Roboflow version", "Value": (data.get("roboflow") or {}).get("version")},
        {"Field": "Roboflow license", "Value": (data.get("roboflow") or {}).get("license")},
        {"Field": "Images", "Value": roboflow.get("images")},
        {"Field": "Classes", "Value": data.get("names")},
        {"Field": "Model", "Value": args.get("model")},
        {"Field": "Epochs configured", "Value": version.get("epochs", args.get("epochs"))},
        {"Field": "Epochs recorded", "Value": metrics.get("epochs_recorded")},
        {"Field": "Image size", "Value": args.get("imgsz")},
        {"Field": "Batch", "Value": args.get("batch")},
        {"Field": "Device", "Value": args.get("device")},
        {"Field": "Patience", "Value": args.get("patience")},
        {"Field": "Best mask mAP50-95", "Value": format_value(metrics.get("best_mask_map50_95"))},
        {"Field": "Best mask mAP50-95 epoch", "Value": metrics.get("best_mask_map50_95_epoch")},
        {"Field": "Final mask mAP50", "Value": format_value(metrics.get("final_mask_map50"))},
        {"Field": "Final mask mAP50-95", "Value": format_value(metrics.get("final_mask_map50_95"))},
        {"Field": "Final box mAP50-95", "Value": format_value(metrics.get("final_box_map50_95"))},
        {"Field": "Best weights", "Value": version["model_path"].relative_to(ROOT)},
    ]
    display_rows(rows)


def display_training_curves(version):
    validate_version_paths(version)
    display(Image.open(version["results_image"]))


def display_mask_overlays(version, conf=0.25, cols=3):
    from ultralytics import YOLO

    validate_version_paths(version)
    image_paths = validation_image_paths(version)
    if not image_paths:
        raise FileNotFoundError(f"No validation images found in {version['valid_images']}")

    model = YOLO(str(version["model_path"]))
    predictions = model.predict(source=[str(p) for p in image_paths], conf=conf, save=False, verbose=False)

    rows = int(np.ceil(len(predictions) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4.5))
    axes = np.atleast_1d(axes).reshape(rows, cols)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, result, image_path in zip(axes.ravel(), predictions, image_paths):
        image = Image.open(image_path)
        masks = result.masks.data if result.masks is not None else None
        overlay = overlay_masks(image, masks)
        mask_count = 0 if masks is None else len(masks)
        ax.imshow(overlay)
        ax.set_title(f"{image_path.name}\n{mask_count} masks", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    display(fig)
    plt.close(fig)


## Version Registry


In [ ]:
V1_DATASET_DIR = ROOT / "datasets" / "sam3_annotation_without_open_source_rebar_v1"
V1_RUN_DIR = ROOT / "rebar-segementation-yolo26" / "yolo26l_sam3_rebar_v1"

VERSIONS = [
    {
        "version": "v1",
        "label": "YOLO26L SAM3 Rebar v1",
        "epochs": 100,
        "dataset_dir": V1_DATASET_DIR,
        "run_dir": V1_RUN_DIR,
        "model_path": V1_RUN_DIR / "weights" / "best.pt",
        "results_image": V1_RUN_DIR / "results.png",
        "results_csv": V1_RUN_DIR / "results.csv",
        "args_yaml": V1_RUN_DIR / "args.yaml",
        "data_yaml": V1_DATASET_DIR / "data.yaml",
        "roboflow_readme": V1_DATASET_DIR / "README.roboflow.txt",
        "valid_images": V1_DATASET_DIR / "valid" / "images",
    }
]

for version in VERSIONS:
    validate_version_paths(version)

VERSION_BY_NAME = {version["version"]: version for version in VERSIONS}
v1 = VERSION_BY_NAME["v1"]


## v1: YOLO26L SAM3 Rebar


### Training Info


In [ ]:
display_training_info(v1)


### Training Curves


In [ ]:
display_training_curves(v1)


### Predicted Mask Overlays


In [ ]:
display_mask_overlays(v1)
